In [3]:
!pip install plotly
!pip install streamlit

  Using cached attrs-25.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached referencing-0.36.2-py3-none-any.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ----------------------------------- ---- 8.9/10.1 MB 46.0 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 30.6 MB/s  0:00:00
   ---------------------------------------- 0.0/731.2 kB ? eta -:--:--
   ---------------------------------------- 731.2/731.2 kB 4.0 MB/s  0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ------------------------------ --------- 5.2/6.9 MB 28.1 MB/s eta 0:00:01
   ---------------------------------------- 6.9/6.9 MB 18.3 MB/s  0:00:00
Using cached attrs-25.3.0-py3-none-any.whl (63 kB)
   ---------------------------------------- 0.0/26.1 MB ? eta -:--:--
   ---- ----------------------------------- 3.1/26.1 MB 14.8 MB/s eta 0:00:02
   -------- ------------------------------- 5.2/26.1 MB 12.5 MB/s eta 0:00:02


In [ ]:
# app.py — Streamlit Portfolio Dashboard 
# Run: streamlit run app.py

import datetime as dt
import io
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

# ----------------------- Page config -----------------------
st.set_page_config(page_title="Portfolio Dashboard", page_icon="📈", layout="wide", initial_sidebar_state="expanded")

# ----------------------- Constants -------------------------
ANNUALIZATION = {"D": 252, "W": 52, "M": 12}

# ----------------------- Utils -----------------------------
def _resample_prices(prices: pd.DataFrame, freq: str) -> pd.DataFrame:
    if not isinstance(prices.index, pd.DatetimeIndex):
        prices = prices.copy()
        prices.index = pd.to_datetime(prices.index)
    if freq == "D":
        return prices.asfreq("B").ffill().dropna(how="all")  # align to business days
    rule = {"W": "W-FRI", "M": "M"}[freq]
    return prices.resample(rule).last().dropna(how="all")

def _returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().dropna(how="all")

def _portfolio_metrics(
    returns: pd.DataFrame, weights: Optional[pd.Series], rf_annual: float, freq: str
) -> Tuple[pd.Series, pd.Series, pd.Series, pd.Series, pd.DataFrame]:
    ann = ANNUALIZATION[freq]
    if (weights is None) or (len(weights) == 0):
        weights = pd.Series(1.0 / returns.shape[1], index=returns.columns)
    else:
        weights = weights.reindex(returns.columns).fillna(0.0)
        if not np.isclose(weights.sum(), 1.0):
            weights = weights / weights.sum()

    # asset stats (annualized)
    mean_a = returns.mean() * ann
    vol_a = returns.std(ddof=0) * np.sqrt(ann)
    sharpe_a = (mean_a - rf_annual) / vol_a.replace(0, np.nan)

    # portfolio series
    port_r = (returns * weights).sum(axis=1)
    port_curve = (1 + port_r).cumprod()
    cum_each = (1 + returns).cumprod() - 1

    # portfolio metrics
    port_vol = port_r.std(ddof=0) * np.sqrt(ann)
    port_ret_ann = port_r.mean() * ann
    port_sharpe = (port_ret_ann - rf_annual) / (port_vol if port_vol != 0 else np.nan)

    return mean_a, vol_a, sharpe_a, port_r, cum_each

def _add_ma(series: pd.Series, sma: int, ema: int) -> pd.DataFrame:
    df = pd.DataFrame({"Price": series})
    if sma > 0:
        df[f"SMA{sma}"] = series.rolling(sma).mean()
    if ema > 0:
        df[f"EMA{ema}"] = series.ewm(span=ema, adjust=False).mean()
    return df

@st.cache_data(show_spinner=False)
def _fetch_yf(tickers: List[str], start: dt.date, end: dt.date, field: str = "Adj Close") -> pd.DataFrame:
    try:
        import yfinance as yf
    except Exception as e:
        raise ImportError("yfinance not available") from e

    df = yf.download(tickers=tickers, start=start, end=end, auto_adjust=False, progress=False, threads=True)
    if df.empty:
        return pd.DataFrame()

    # Multi-index columns if multiple tickers
    if isinstance(df.columns, pd.MultiIndex):
        if field in df.columns.get_level_values(0):
            dfp = df[field].copy()
        else:
            # fallback: prefer Close
            fallback = "Close" if "Close" in df.columns.get_level_values(0) else df.columns.levels[0][0]
            dfp = df[fallback].copy()
    else:
        # Single ticker
        dfp = df.rename(columns={df.columns[0]: tickers[0]}).copy()

    return dfp.dropna(how="all")

def _synth_prices(tickers: List[str], start: dt.date, end: dt.date) -> pd.DataFrame:
    # Deterministic synthetic prices (geometric random walk) so the app always runs.
    idx = pd.bdate_range(start, end, inclusive="both")
    np.random.seed(42)
    prices = {}
    for i, t in enumerate(tickers):
        mu, sigma = 0.10, 0.25  # annualized assumptions
        n = len(idx)
        # convert to daily
        mu_d = mu / 252.0
        sigma_d = sigma / np.sqrt(252.0)
        r = np.random.normal(loc=mu_d, scale=sigma_d, size=n)
        p = 100.0 * np.cumprod(1 + r)
        prices[str(t)] = p
    return pd.DataFrame(prices, index=idx)

def _download_button(df: pd.DataFrame, label: str, filename: str):
    buf = io.BytesIO()
    df.to_csv(buf)
    st.download_button(label=label, data=buf.getvalue(), file_name=filename, mime="text/csv", use_container_width=True)

# ----------------------- Sidebar Controls ------------------
st.sidebar.title("⚙️ Controls")

data_source = st.sidebar.radio("Data Source", ("Yahoo Finance (auto / fallback)", "Upload CSV"))
default_tickers = ["AAPL", "MSFT", "NVDA", "SPY", "BTC-USD"]

today = dt.date.today()
start = st.sidebar.date_input("Start Date", value=today - dt.timedelta(days=365 * 3))
end = st.sidebar.date_input("End Date", value=today)

freq = st.sidebar.selectbox("Frequency", ["D", "W", "M"], index=0)
rf = st.sidebar.number_input("Risk-free rate (annual, e.g., 0.02 = 2%)", value=0.02, format="%.6f")

sma_w = st.sidebar.number_input("SMA window", min_value=0, value=50, step=5)
ema_w = st.sidebar.number_input("EMA window", min_value=0, value=20, step=5)

st.sidebar.markdown("---")
st.sidebar.caption("Upload weights (optional): two columns named **Asset,Weight** that sum to 1.")
weights_file = st.sidebar.file_uploader("Weights CSV (optional)", type=["csv"])

weights = None
if weights_file is not None:
    try:
        wdf = pd.read_csv(weights_file)
        cols = {c.lower(): c for c in wdf.columns}
        asset_col, weight_col = cols.get("asset"), cols.get("weight")
        if asset_col and weight_col:
            weights = pd.Series(wdf[weight_col].values, index=wdf[asset_col].astype(str).values)
        else:
            st.sidebar.error("Weights CSV must have columns: Asset, Weight")
    except Exception as e:
        st.sidebar.error(f"Failed to read weights: {e}")

# Data inputs
uploaded_prices = None
if data_source == "Upload CSV":
    uploaded_file = st.sidebar.file_uploader("Prices CSV (Date in 1st col; others = assets)", type=["csv"])
    if uploaded_file is not None:
        try:
            dfu = pd.read_csv(uploaded_file)
            date_col = dfu.columns[0]
            dfu[date_col] = pd.to_datetime(dfu[date_col])
            dfu = dfu.set_index(date_col).sort_index()
            # Clip to range
            dfu = dfu.loc[(dfu.index.date >= start) & (dfu.index.date <= end)]
            uploaded_prices = dfu.copy()
        except Exception as e:
            st.sidebar.error(f"Failed to parse CSV: {e}")
else:
    ticker_str = st.sidebar.text_input("Tickers (comma-separated)", value=",".join(default_tickers))
    tickers = [t.strip() for t in ticker_str.split(",") if t.strip()]
    price_field = st.sidebar.selectbox("Price Field", ["Adj Close", "Close"], index=0)

# ----------------------- Main ------------------------------
st.title("📊 Portfolio Dashboard (Streamlit)")
st.caption("Daily & cumulative returns, volatility, Sharpe, and moving averages with interactive controls.")

# Acquire prices (no buttons; auto and robust)
prices = pd.DataFrame()
if data_source == "Upload CSV":
    if uploaded_prices is None or uploaded_prices.empty:
        st.info("➡️ Upload a prices CSV to proceed. Expected wide format with dates in the first column.")
        st.stop()
    prices = uploaded_prices.copy()
else:
    # Try Yahoo first; if it fails, use synthetic so the app always runs
    try:
        prices = _fetch_yf(tickers, start, end, field=price_field)
        if prices.empty:
            raise RuntimeError("Empty data from Yahoo")
    except Exception:
        prices = _synth_prices(tickers, start, end)
        st.warning("Using **synthetic prices** (Yahoo Finance unavailable). You can still explore the dashboard.")

# Clean + resample
prices = prices.dropna(how="all")
if prices.shape[1] == 0:
    st.warning("No valid assets after cleaning. Please adjust inputs.")
    st.stop()

prices = _resample_prices(prices, freq)

# Ensure enough data
if prices.shape[0] < 2:
    st.warning("Not enough data points to compute returns. Try a longer window.")
    st.stop()

rets = _returns(prices)
if rets.empty:
    st.warning("Returns are empty after differencing. Try different data/frequency.")
    st.stop()

# Metrics
mean_a, vol_a, sharpe_a, port_r, cum_each = _portfolio_metrics(rets, weights, rf, freq)
port_curve = (1 + port_r).cumprod()
port_cum = port_curve - 1
port_vol_ann = port_r.std(ddof=0) * np.sqrt(ANNUALIZATION[freq])
port_sharpe = (port_r.mean() * ANNUALIZATION[freq] - rf) / (port_vol_ann if port_vol_ann != 0 else np.nan)

# ----------------------- KPIs ------------------------------
k1, k2, k3, k4 = st.columns(4)
with k1:
    st.metric("Portfolio Cumulative Return", f"{(port_curve.iloc[-1] - 1):.2%}")
with k2:
    st.metric(f"Volatility ({freq})", f"{port_vol_ann:.2%}")
with k3:
    st.metric("Sharpe Ratio", f"{port_sharpe:.2f}")
with k4:
    st.metric("Assets Tracked", f"{prices.shape[1]}")

# ----------------------- Charts ----------------------------
tab1, tab2, tab3 = st.tabs(["📈 Prices & MAs", "📈 Cumulative Returns", "📉 Drawdown"])

with tab1:
    left, right = st.columns([2, 1])
    with right:
        ma_asset = st.selectbox("Asset for MA overlay", options=list(prices.columns), index=0)
    with left:
        ma_df = _add_ma(prices[ma_asset].dropna(), sma_w, ema_w)
        fig = go.Figure()
        for col in ma_df.columns:
            fig.add_trace(go.Scatter(x=ma_df.index, y=ma_df[col], mode="lines", name=col))
        fig.update_layout(
            title=f"{ma_asset} Price with Moving Averages",
            xaxis_title="Date",
            yaxis_title="Price",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        )
        st.plotly_chart(fig, use_container_width=True)

with tab2:
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=port_cum.index, y=port_cum.values, mode="lines", name="Portfolio"))
    for col in cum_each.columns:
        fig2.add_trace(go.Scatter(x=cum_each.index, y=cum_each[col], mode="lines", name=str(col), opacity=0.5))
    fig2.update_layout(
        title="Cumulative Returns",
        xaxis_title="Date",
        yaxis_title="Cumulative Return",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    st.plotly_chart(fig2, use_container_width=True)

with tab3:
    roll_max = port_curve.cummax()
    drawdown = port_curve / roll_max - 1
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=drawdown.index, y=drawdown.values, mode="lines", name="Drawdown"))
    fig3.update_layout(
        title="Portfolio Drawdown",
        xaxis_title="Date",
        yaxis_title="Drawdown",
        yaxis=dict(tickformat=".0%"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    st.plotly_chart(fig3, use_container_width=True)

# ----------------------- Tables ----------------------------
st.subheader("📋 Risk–Return Metrics (Annualized)")
metrics_df = pd.DataFrame({"Mean Return": mean_a, "Volatility": vol_a, "Sharpe": sharpe_a}).sort_index()
st.dataframe(metrics_df.style.format({"Mean Return": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.2f}"}), use_container_width=True)

st.markdown("**Portfolio Summary**")
port_summary = pd.DataFrame(
    {"Mean Return": [port_r.mean() * ANNUALIZATION[freq]], "Volatility": [port_vol_ann], "Sharpe": [port_sharpe]},
    index=["Portfolio"],
)
st.dataframe(port_summary.style.format({"Mean Return": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.2f}"}), use_container_width=True)

# ----------------------- Allocation Pie --------------------
st.subheader("📎 Allocation (by Asset)")
if (weights is None) or (len(weights) == 0):
    w = pd.Series(1.0 / prices.shape[1], index=prices.columns)
else:
    w = weights.reindex(prices.columns).fillna(0.0)
    if not np.isclose(w.sum(), 1.0):
        w = w / w.sum()

fig_alloc = go.Figure(data=[go.Pie(labels=w.index.astype(str), values=w.values, hole=0.4)])
fig_alloc.update_layout(title_text="Portfolio Allocation", showlegend=True)
st.plotly_chart(fig_alloc, use_container_width=True)

# ----------------------- Downloads -------------------------
st.subheader("⬇️ Downloads")
c1, c2, c3 = st.columns(3)
with c1:
    _download_button(prices, "Download Prices CSV", "prices.csv")
with c2:
    _download_button(rets, "Download Returns CSV", "returns.csv")
with c3:
    _download_button(metrics_df, "Download Risk-Return Metrics CSV", "risk_return_metrics.csv")

# ----------------------- Notes -----------------------------
with st.expander("Notes & Tips"):
    st.markdown(
        """
- **Sharpe** uses annualization based on your chosen frequency (D=252, W=52, M=12) and the **annual** risk-free rate.
- **Weights** default to equal. Upload a two-column CSV (`Asset,Weight`) to customize; weights will be normalized if they don't sum to 1.
- If Yahoo Finance is unavailable, the app switches to **synthetic prices** so you can still interact with the UI.
- CSV upload expects wide prices: first column = date, remaining columns = assets.
        """
    )


2025-09-29 16:46:09.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.132 No runtime found, using MemoryCacheStorageManager
2025-09-29 16:46:09.137 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.138 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.139 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.140 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.141 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.142 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.144 Thread 'MainThread':

2025-09-29 16:46:09.161 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.162 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.162 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.163 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.164 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.165 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.166 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-29 16:46:09.167 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar